In [31]:
import polars as pl
import numpy as np
import os

# Load PONSOL dataset
df = pl.read_csv(
    "../data/ponsol.csv",
    has_header=True,
)

# Select relevant columns and drop rows with missing target values
df = df.select(["sequence", "mutations", "solubility_change"]).drop_nulls(["solubility_change"])

df

sequence,mutations,solubility_change
str,str,f64
"""DVSGTVCLSALPPEATDTLNLIASDGPFPY…","""T76I""",-0.59
"""MPSSVSWGILLLAGLCCLVPVSLAEDPQGD…","""M382L""",-0.149123
"""MAEVPELASEMMAYYSGNEDDLFFEADGPK…","""L126N""",-0.554348
"""MAEVPELASEMMAYYSGNEDDLFFEADGPK…","""L126D""",-0.76087
"""MAEVPELASEMMAYYSGNEDDLFFEADGPK…","""L126T""",-0.815217
…,…,…
"""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…","""L83K""",-1.0
"""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…","""H85I""",0.0
"""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…","""H85M""",-0.1


In [32]:
# Extract mutation details into separate columns (e.g., "T76I" -> from: T, pos: 76, to: I)
df = df.with_columns([
    pl.col("mutations").str.extract(r'([A-Za-z])(\d+)([A-Za-z])', 1).alias("mutation_from"),
    pl.col("mutations").str.extract(r'([A-Za-z])(\d+)([A-Za-z])', 2).cast(pl.Int64).alias("mutation_pos"),
    pl.col("mutations").str.extract(r'([A-Za-z])(\d+)([A-Za-z])', 3).alias("mutation_to"),
    pl.col("sequence").str.len_chars().alias("seq_len"),
])

# Verify amino acid at mutation position and generate mutated sequence
df = df.with_columns([
    pl.col("sequence").str.slice(pl.col("mutation_pos") - 1, 1).alias("seq_aa_at_pos")])

df = df.with_columns([
    pl.when(
        (pl.col("mutation_pos").is_not_null())
        & (pl.col("mutation_pos") >= 1)
        & (pl.col("mutation_pos") <= pl.col("seq_len"))
        & (pl.col("seq_aa_at_pos") == pl.col("mutation_from")),
    ).then(
        # Construct mutated sequence: prefix + new AA + suffix
        pl.col("sequence").str.slice(0, pl.col("mutation_pos") - 1)
        + pl.col("mutation_to")
        + pl.col("sequence").str.slice(pl.col("mutation_pos"), pl.col("seq_len") - pl.col("mutation_pos"))
    ).otherwise(None).alias("mut_sequence")

]).rename({
    "sequence": "wt_sequence",
    "mutations": "mutation"
}).select([
    "wt_sequence",
    "mutation",
    "mut_sequence",
    "solubility_change"
])

# Display summary statistics for solubility change
df.select(pl.col("solubility_change")).describe()

statistic,solubility_change
str,f64
"""count""",133.0
"""null_count""",0.0
"""mean""",-0.299247
"""std""",0.630223
"""min""",-1.0
"""25%""",-0.7
"""50%""",-0.3
"""75%""",-0.1
"""max""",4.88745


In [33]:
# Normalize solubility change using sigmoid function (similar to S350 processing)
k_neg = 3.5
k_pos = 0.8
A_neg = 1.0
A_pos = 1.0

sigmoid_expr = (
    pl.when(pl.col("solubility_change") >= 0)
    .then(A_pos * (2 / (1 + (-k_pos * pl.col("solubility_change")).exp()) - 1))
    .otherwise(-A_neg * (2 / (1 + (-k_neg * (-pl.col("solubility_change"))).exp()) - 1))
)

df = df.with_columns(
    sigmoid_expr.alias("target")
)
df

wt_sequence,mutation,mut_sequence,solubility_change,target
str,str,str,f64,f64
"""DVSGTVCLSALPPEATDTLNLIASDGPFPY…","""T76I""","""DVSGTVCLSALPPEATDTLNLIASDGPFPY…",-0.59,-0.774909
"""MPSSVSWGILLLAGLCCLVPVSLAEDPQGD…","""M382L""","""MPSSVSWGILLLAGLCCLVPVSLAEDPQGD…",-0.149123,-0.255198
"""MAEVPELASEMMAYYSGNEDDLFFEADGPK…","""L126N""","""MAEVPELASEMMAYYSGNEDDLFFEADGPK…",-0.554348,-0.748752
"""MAEVPELASEMMAYYSGNEDDLFFEADGPK…","""L126D""","""MAEVPELASEMMAYYSGNEDDLFFEADGPK…",-0.76087,-0.869621
"""MAEVPELASEMMAYYSGNEDDLFFEADGPK…","""L126T""","""MAEVPELASEMMAYYSGNEDDLFFEADGPK…",-0.815217,-0.890974
…,…,…,…,…
"""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…","""L83K""","""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…",-1.0,-0.941376
"""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…","""H85I""","""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…",0.0,0.0
"""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…","""H85M""","""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…",-0.1,-0.173235


In [34]:
# Save prepared dataset to Parquet
if not os.path.exists("datasets"):
    os.makedirs("datasets")
df.write_parquet("datasets/ponsol_prepared.parquet")

In [36]:
df.filter(pl.col("target") >= 0)

wt_sequence,mutation,mut_sequence,solubility_change,target
str,str,str,f64,f64
"""MAEVPELASEMMAYYSGNEDDLFFEADGPK…","""K213R""","""MAEVPELASEMMAYYSGNEDDLFFEADGPK…",0.076087,0.030425
"""MMEQVCDVFDIYAICACCKVESKNEGKKNE…","""K27E""","""MMEQVCDVFDIYAICACCKVESKNEGEKNE…",4.88745,0.960705
"""DVSGTVCLSALPPEATDTLNLIASDGPFPY…","""T76K""","""DVSGTVCLSALPPEATDTLNLIASDGPFPY…",0.55,0.216518
"""DVSGTVCLSALPPEATDTLNLIASDGPFPY…","""T76S""","""DVSGTVCLSALPPEATDTLNLIASDGPFPY…",0.95,0.362707
"""DVSGTVCLSALPPEATDTLNLIASDGPFPY…","""T76A""","""DVSGTVCLSALPPEATDTLNLIASDGPFPY…",0.35,0.139092
…,…,…,…,…
"""MAEVPELASEMMAYYSGNEDDLFFEADGPK…","""T125W""","""MAEVPELASEMMAYYSGNEDDLFFEADGPK…",0.0434783,0.01739
"""MAEVPELASEMMAYYSGNEDDLFFEADGPK…","""E212G""","""MAEVPELASEMMAYYSGNEDDLFFEADGPK…",0.0217391,0.008695
"""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…","""H55V""","""MQFKVYTYKRESRYRLFVDVQSDIIDTPGR…",0.0,0.0


In [37]:
df.select(["wt_sequence"]).with_columns(pl.col("wt_sequence").str.len_chars().alias("length")).describe()

statistic,wt_sequence,length
str,str,f64
"""count""","""133""",133.0
"""null_count""","""0""",0.0
"""mean""",null,155.165414
"""std""",null,97.435463
"""min""","""DVSGTVCLSALPPEATDTLNLIASDGPFPY…",96.0
"""25%""",null,101.0
"""50%""",null,101.0
"""75%""",null,204.0
"""max""","""VAEKAKDERELLEKTSELIAGMGDKIGEHL…",608.0
